# LLMToolEmulatorMiddleware：用 LLM 模拟工具执行

`LLMToolEmulatorMiddleware`（实际类名为 `LLMToolEmulator`）会在 Agent 的工具执行阶段拦截指定工具，用另一个 LLM 生成“看起来像工具返回值”的内容，并把该内容作为 `ToolMessage` 交回给 Agent。

本节目标：

1. 理解它拦截的是**工具执行**，而不是工具选择。
2. 掌握 `tools` 与 `model` 两个参数，以及“全部模拟”和“选择性模拟”的区别。
3. 用可观察的调用日志证明：被模拟的工具函数没有真正执行。
4. 明确它适合原型与 Agent 行为测试，但不能替代确定性的单元测试或真实集成测试。


## 1. 机制与执行位置

Agent 的主模型仍会根据工具名称、描述与用户问题，决定是否发起工具调用；`LLMToolEmulator` 不会替模型挑选工具，也不会从 Agent 的工具列表中移除工具。它实现的是 Middleware 的 `wrap_tool_call` / `awrap_tool_call` 钩子：

```text
用户请求
    -> Agent 主模型决定调用某个工具
    -> LLMToolEmulator 拦截该工具调用
        -> 该工具在模拟范围内？
            -> 是：调用模拟 LLM，生成 ToolMessage；真实工具函数不执行
            -> 否：调用 handler，真实工具函数照常执行
    -> Agent 主模型读取 ToolMessage，继续推理或生成最终回答
```

模拟 LLM 接收到的信息包括：工具名称、工具描述和本次工具参数。它被要求只返回一个“符合该工具返回格式的、合理的结果”。因此模拟的是**工具返回内容**，不是模拟 Agent 的工具调用决策。

同步 Agent 调用 `invoke()` 时，Middleware 内部调用模拟模型的 `invoke()`；异步 Agent 调用 `ainvoke()` 时，它会调用模拟模型的 `ainvoke()`。


## 2. 构造参数与版本注意事项

```python
LLMToolEmulator(
    tools: list[str | BaseTool] | None = None,
    model: str | BaseChatModel | None = None,
)
```

| 参数 | 含义 | 关键行为 |
| --- | --- | --- |
| `tools=None` | 默认值 | 模拟 Agent 中的**所有**工具。 |
| `tools=[]` | 空列表 | 不模拟任何工具，所有工具都会真实执行。一般没有单独配置此 Middleware 的必要。 |
| `tools=["get_weather"]` | 工具名列表 | 只模拟指定名称的工具。 |
| `tools=[get_weather]` | 工具对象列表 | 与传工具名等价，Middleware 会读取工具对象的 `.name`。 |
| `model` | 生成模拟结果的模型 | 可以是模型字符串或已初始化的 `BaseChatModel`。建议显式传入。 |

**版本注意：** 当前项目安装的是 `langchain==1.3.11`。该版本源码中，`model=None` 会默认初始化 `anthropic:claude-sonnet-4-5-20250929`，并设置 `temperature=1`。这会带来额外的 Anthropic 凭证依赖和更明显的随机性。因此本笔记始终显式传入 `emulator_model`，不依赖默认值。

如果传入模型字符串，当前实现仍会以 `temperature=1` 初始化它；如果传入一个已初始化的模型对象，则保留该对象自己的参数。下面把模拟模型初始化为 `temperature=0`，以降低演示结果的随机性，但它仍不是严格确定性的 Mock。


## 3. 适用边界：它不是离线 Mock

LLM Tool Emulator 适用于：

- 外部 API 尚未实现、不可访问或调用昂贵时，先验证 Agent 是否会正确选择和消费工具结果。
- 原型开发阶段，快速跑通“模型选择工具 -> 工具结果 -> 最终回答”的 Agent 闭环。
- 对带副作用的工具做演示，例如发邮件、创建订单；演示时不真正触发副作用。

它不适用于：

- 验证真实 API 的参数、鉴权、网络错误、限流、响应 schema 或业务规则。
- 需要可重复、断言精确返回值的单元测试。
- 生产环境中替代真实工具。

| 方案 | 是否调用真实工具 | 是否调用 LLM | 结果是否确定 | 适用层级 |
| --- | --- | --- | --- |
| 手写 stub / mock | 否 | 否 | 可以严格确定 | 单元测试 |
| `LLMToolEmulator` | 否 | 是 | 通常不确定 | 原型、Agent 行为探索、演示 |
| 测试环境真实工具 | 是（测试资源） | 视 Agent 而定 | 取决于环境 | 集成测试 |


In [ ]:
from importlib.metadata import version
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 从 .env 中加载 OPENROUTER_API_KEY 与 OPENROUTER_BASE_URL。
load_dotenv(override=True)

print("langchain version:", version("langchain"))

# 主模型：负责理解用户请求、选择工具、读取 ToolMessage 后生成最终回答。
agent_model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    temperature=0,
)

# 模拟模型：只负责生成“被模拟工具”的返回值。
# 显式传入可避免 LLMToolEmulator 使用其默认 Anthropic 模型。
emulator_model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    temperature=0,
)


In [ ]:
from langchain.tools import tool
from langchain.messages import HumanMessage, ToolMessage
from rich import print as rprint

# 用日志替代真实副作用：只有真实工具函数被执行时，日志才会增加记录。
real_call_log: list[tuple[str, str]] = []


@tool
def get_weather(city: str) -> str:
    """查询指定城市的实时天气。返回天气、温度和风力。"""
    real_call_log.append(("get_weather", city))
    # 在真实项目中，这里会请求天气 API；本笔记用固定值代替。
    return f"真实天气服务：{city}，晴，23 摄氏度，微风。"


@tool
def check_inventory(product: str) -> str:
    """查询商品库存。返回商品名和当前可用库存数量。"""
    real_call_log.append(("check_inventory", product))
    # 在真实项目中，这里会查询数据库或库存服务。
    return f"真实库存服务：{product} 当前可用库存为 18 件。"


def show_tool_messages(result: dict) -> list[ToolMessage]:
    """打印本次 Agent 运行中返回给模型的工具结果。"""
    tool_messages = [
        message
        for message in result["messages"]
        if isinstance(message, ToolMessage)
    ]
    for message in tool_messages:
        print(f"工具名: {message.name}")
        rprint(message.content)
        print("-" * 50)
    return tool_messages


## 4. 示例一：默认模拟全部工具

`tools=None` 是默认行为，因此下例中 `get_weather` 虽然仍在 Agent 的工具列表里，且主模型仍会选择它，但真正的 Python 函数不会执行。`real_call_log` 应保持为空。

为了让验证更明确，系统提示要求模型必须先调用 `get_weather`，再回答用户。若所用模型没有按指令发起工具调用，代码不会抛出误导性的异常，而会提示检查模型输出。


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator

real_call_log.clear()

all_emulated_agent = create_agent(
    model=agent_model,
    tools=[get_weather],
    middleware=[
        # tools=None：模拟所有工具；显式指定模型，避免使用默认 Anthropic 模型。
        LLMToolEmulator(model=emulator_model),
    ],
    system_prompt=(
        "你是天气助手。回答天气问题前，必须调用 get_weather；"
        "不要根据常识直接编造天气。"
    ),
)

all_emulated_result = all_emulated_agent.invoke(
    {
        "messages": [
            HumanMessage(content="请查询北京现在的天气，并用一句话总结。")
        ]
    }
)

print("=== Agent 收到的工具结果 ===")
all_emulated_tool_messages = show_tool_messages(all_emulated_result)

print("=== 真实工具执行日志 ===")
print(real_call_log)

if all_emulated_tool_messages and not real_call_log:
    print("验证成功：模型调用了工具，但 get_weather 的 Python 函数没有执行。")
else:
    print("未得到预期现象：请检查工具调用记录和模型是否遵循了系统提示。")

print("=== Agent 最终回答 ===")
rprint(all_emulated_result["messages"][-1].content)


## 5. 示例二：只模拟指定工具

下面只模拟 `get_weather`，而 `check_inventory` 保持真实执行。用户请求同时需要两个工具时，预期现象为：

- `get_weather` 会产生 ToolMessage，但不会写入 `real_call_log`。
- `check_inventory` 会产生 ToolMessage，且会写入 `real_call_log`。

选择性模拟适合“部分外部服务尚不可用，但其余工具仍希望真实联调”的场景。除了字符串名称，`tools=[get_weather]` 也能表达同样的选择。


In [ ]:
real_call_log.clear()

selectively_emulated_agent = create_agent(
    model=agent_model,
    tools=[get_weather, check_inventory],
    middleware=[
        LLMToolEmulator(
            tools=["get_weather"],
            model=emulator_model,
        )
    ],
    system_prompt=(
        "你是商品助手。涉及天气时必须调用 get_weather；"
        "涉及库存时必须调用 check_inventory；不要直接猜测结果。"
    ),
)

selectively_emulated_result = selectively_emulated_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="北京天气如何？另外请查询机械键盘的库存。"
            )
        ]
    }
)

print("=== Agent 收到的工具结果 ===")
selectively_emulated_tool_messages = show_tool_messages(selectively_emulated_result)

print("=== 真实工具执行日志 ===")
print(real_call_log)

tool_names = {message.name for message in selectively_emulated_tool_messages}
weather_was_real = any(name == "get_weather" for name, _ in real_call_log)
inventory_was_real = any(name == "check_inventory" for name, _ in real_call_log)

if {"get_weather", "check_inventory"}.issubset(tool_names):
    print("两个工具都被主模型调用。")
else:
    print("模型没有调用全部预期工具，请先查看上方工具结果。")

if not weather_was_real and inventory_was_real:
    print("验证成功：天气工具被模拟，库存工具真实执行。")
else:
    print("选择性模拟结果未完全符合预期，请检查真实执行日志。")

print("=== Agent 最终回答 ===")
rprint(selectively_emulated_result["messages"][-1].content)


## 6. 重要限制与测试策略

1. **不是零成本。** 每个被模拟的工具调用都会额外调用一次 `emulator_model`。它节省的是外部工具调用与副作用，不一定节省 LLM token 或总延迟。
2. **不是严格 Mock。** 当前实现提示模拟 LLM“生成真实且带变化的结果”。即使温度设为 0，不同模型版本、服务端设置或提示上下文仍可能造成差异；不要断言某个固定模拟文本。
3. **仍有数据暴露面。** 工具名称、描述与调用参数会发送给模拟模型。即使真实工具未运行，也不能把生产密钥、个人数据或敏感业务参数随意放进模拟调用。
4. **无法验证真实工具契约。** 它不会发现真实 API 的 schema 改动、认证失败、网络超时、限流、数据库事务问题或真实副作用。
5. **合理的测试分层：** 用 stub / mock 做确定性单元测试；用 LLMToolEmulator 快速探索 Agent 行为；用隔离测试环境中的真实工具做集成测试；生产环境则关闭模拟并加入权限、审计和限流。

因此，最合适的定位是：**用自然语言生成“足够像真的”工具反馈，测试 Agent 是否能把工具调用闭环走通，而不是证明业务系统本身正确。**


## 7. 小结与官方参考

常用配置可以归纳为：

```python
# 原型阶段：模拟所有工具
LLMToolEmulator(model=emulator_model)

# 局部联调：只模拟不稳定、昂贵或有副作用的工具
LLMToolEmulator(tools=["send_email", "create_order"], model=emulator_model)

# 传入工具对象也可以
LLMToolEmulator(tools=[send_email], model=emulator_model)
```

使用时记住两句话：

- 它保留“主模型选择工具”的真实行为，只替换“工具执行与返回”的环节。
- 显式传入模拟模型；不要依赖默认模型、默认凭证或随机输出。

官方参考：

- [LangChain Prebuilt middleware - LLM tool emulator](https://docs.langchain.com/oss/python/langchain/middleware/built-in)
- [LLMToolEmulator API Reference](https://reference.langchain.com/python/langchain/agents/middleware/tool_emulator/LLMToolEmulator)
